# Christenson: Chapter Titles

In this notebook, we prepare a dataframe of chapter titles from Christenson's literal english translation. Each chapter is numbered and associated with page numbers in the book as well as beginning and ending line numbers is Christenson's transcription of the _Popol Wuj_. 

Note that Christenson misnumbers lines in his literal translation:

- The line marked $4910$ on page $170$ is actually line $4908$, leaving $8$ lines in one group and $12$ in the next.
- Line $5250$ repeated twice on page $182$, throwing off the rest by $10$. This means the total number of lines in the text is $8726$, not $8716$.

Therefore, the `xom-` lines below need to be corrected if we want to map to source.

In [1]:
import pandas as pd
import numpy as np

The titles file below was extracted from Christenson's literal english translation. This is essentially a table of content.

In [2]:
titles = pd.read_csv("christenson-pv-titles.csv", sep="|")
titles.index.name = 'chap_num'
titles.index += 1

We import data generated by `02-christenson-grab-line-counts.py`.

This script extracts title and associated line numbers from `christenson-pv-english.txt`.

# Grab Line Numbers for Chapters

In [52]:
import re

lines = open("christenson-pv-english.txt", "r").readlines()
xom_lines_list = []
for line in lines:
    if re.match(r"^\s+\d+\s+(L|l)ines\s+", line):
        xom_lines_list.append(line.strip().split()[-1])
xom_lines = pd.DataFrame(xom_lines_list, columns=['xom_lines'])
xom_lines.index.name = 'chap_num'
xom_lines.index += 1
xom_lines

,xom_lines
chap_num,
1,1-96
2,97-154
3,155-274
4,275-339
5,340-433
...,...
83,8596-8631
84,8632-8663
85,8664-8679


In [53]:
CHAP = titles.join(xom_lines)
CHAP

,chap_title,page_num,xom_lines
chap_num,,,
1,Preamble,59,1-96
2,The Primordial World,67,97-154
3,The Creation of the Earth,70,155-274
4,The Creation of the Animals,74,275-339
5,The Fall of the Animals,76,340-433
...,...,...,...
83,The Dynasty of Nihaib Lords,301,8596-8631
84,The Great Houses of the Nihaib Lords,302,8632-8663
85,The Dynasty of Ahau Quiché Lords,303,8664-8679


In [55]:
if 'xom_start' not in CHAP.columns:
    CHAP = CHAP.join(CHAP.xom_lines.str.split('-')\
        .apply(pd.Series).rename(columns={0:'xom_start',1:'xom_end'}))
    CHAP.xom_start = CHAP.xom_start.astype(int)
    CHAP.xom_end = CHAP.xom_end.astype(int)
    CHAP = CHAP.drop('xom_lines', axis=1)
CHAP

,chap_title,page_num,xom_start,xom_end
chap_num,,,,
1,Preamble,59,1,96
2,The Primordial World,67,97,154
3,The Creation of the Earth,70,155,274
4,The Creation of the Animals,74,275,339
5,The Fall of the Animals,76,340,433
...,...,...,...,...
83,The Dynasty of Nihaib Lords,301,8596,8631
84,The Great Houses of the Nihaib Lords,302,8632,8663
85,The Dynasty of Ahau Quiché Lords,303,8664,8679


# Fix the Offset

$5250$ is repeated twice.

So, find the chapter where it exists and add $10$ from that point on.

In [56]:
CHAP[CHAP.xom_start > 5250].head()

,chap_title,page_num,xom_start,xom_end
chap_num,,,,
48,The Arrival at Tulan,209,5361,5374
49,The Progenitors Receive their Gods,211,5375,5413
50,The Progenitors Leave Tulan,213,5414,5485
51,The Nations Ask for their Fire,214,5486,5601
52,The Nations Are Deceived into Offering Themselves,216,5602,5774


In [57]:
CHAP.loc[47]

chap_title    The Creation of the Mothers of the Quiché Nation
page_num                                                   201
xom_start                                                 5116
xom_end                                                   5360
Name: 47, dtype: object

Create new columns.

So, the errors begin at:

- xom_start: $>= 5361$
- xom_end: $>= 5360$

In [58]:
CHAP['xom_start_fixed'] = CHAP.xom_start
CHAP['xom_end_fixed'] = CHAP.xom_end
CHAP.loc[CHAP.xom_start >= 5361, 'xom_start_fixed'] = CHAP.xom_start + 10
CHAP.loc[CHAP.xom_end >= 5360, 'xom_end_fixed'] = CHAP.xom_end + 10
CHAP

,chap_title,page_num,xom_start,xom_end,xom_start_fixed,xom_end_fixed
chap_num,,,,,,
1,Preamble,59,1,96,1,96
2,The Primordial World,67,97,154,97,154
3,The Creation of the Earth,70,155,274,155,274
4,The Creation of the Animals,74,275,339,275,339
5,The Fall of the Animals,76,340,433,340,433
...,...,...,...,...,...,...
83,The Dynasty of Nihaib Lords,301,8596,8631,8606,8641
84,The Great Houses of the Nihaib Lords,302,8632,8663,8642,8673
85,The Dynasty of Ahau Quiché Lords,303,8664,8679,8674,8689


# Save

In [59]:
CHAP.to_csv("christenson-CHAP.csv", index=True)